# Phase 1 at full scale on Kaggle — Bengali 500M tokens + Nepali

Runs the same pipeline code as the local `sampled` profile with the caps removed.

## Why Kaggle suits this better than Colab

| | Kaggle | Colab (free) |
|---|---|---|
| RAM | **30 GB** | ~13 GB |
| Session | 12 h, no idle disconnect while running | disconnects readily |
| Persistence | `/kaggle/working` survives between sessions of the same notebook | needs Drive mount |

RAM is the deciding factor. This ingest holds a MinHash deduplication signature
per kept document; at ~1.4M documents that index is the largest thing in memory,
and it is what makes a 7 GB laptop fail. 30 GB removes the constraint entirely.

## Before you run anything

In the notebook sidebar (right-hand panel):

1. **Settings → Accelerator → None**. This stage is CPU and I/O bound; a GPU does
   nothing for it and consumes your weekly GPU quota, which Phase 2 needs.
2. **Settings → Internet → On**. Required to reach the Hugging Face hub. Kaggle
   disables internet by default and this is the single most common reason the
   dataset cells fail.
3. **Add-ons → Secrets** — add `GH_TOKEN` (and optionally `HF_TOKEN`), described below.


## 1 · Environment check

Confirms the settings above actually took effect, before anything long-running starts.


In [ ]:
import os, sys, shutil, subprocess, pathlib

WORK = pathlib.Path('/kaggle/working')
REPO = WORK / 'lma'

print('python      :', sys.version.split()[0])
print('cpus        :', os.cpu_count())

with open('/proc/meminfo') as fh:
    total_kb = int(fh.readline().split()[1])
print(f'memory      : {total_kb / 1024**2:.1f} GB')

usage = shutil.disk_usage(WORK)
print(f'working disk: {usage.free / 1e9:.1f} GB free of {usage.total / 1e9:.1f} GB')

# Internet is off by default on Kaggle; check it rather than discover it later.
probe = subprocess.run(['curl', '-sS', '-m', '10', '-o', '/dev/null',
                        '-w', '%{http_code}', 'https://huggingface.co'],
                       capture_output=True, text=True)
online = probe.stdout.strip() == '200'
print('internet    :', 'ON' if online else 'OFF  <-- enable it in Settings -> Internet')
assert online, 'Enable Internet in the notebook settings panel, then re-run this cell.'


## 2 · Secrets

**Add-ons → Secrets → Add a new secret** in the notebook menu.

| Label | Value | Needed for |
|---|---|---|
| `GH_TOKEN` | GitHub personal access token, **classic**, `repo` scope | cloning your private Classroom repo |
| `HF_TOKEN` | Hugging Face read token | higher hub rate limits (optional) |

For the GitHub token use **Tokens (classic)** with the top-level `repo` box ticked.
A fine-grained token defaults to no repository access and fails with a misleading
`403 Write access to repository not granted`.

Secrets stay out of the notebook source, so the notebook is safe to submit or share.


In [ ]:
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

def get_secret(label):
    """Read a Kaggle secret, returning None when it is not set."""
    try:
        return secrets.get_secret(label)
    except Exception:
        return None

GH_TOKEN = get_secret('GH_TOKEN')
HF_TOKEN = get_secret('HF_TOKEN')

print('GH_TOKEN:', 'set' if GH_TOKEN else 'MISSING - the clone below will fail')
print('HF_TOKEN:', 'set' if HF_TOKEN else 'not set (unauthenticated, rate-limited)')

if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN

# Cache datasets under /kaggle/working so a re-run in the same notebook does not
# re-download shards already fetched.
os.environ['HF_HOME'] = str(WORK / 'hf_cache')


from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()

def get_secret(label):
    """Read a Kaggle secret, returning None when it is not set."""
    try:
        return secrets.get_secret(label)
    except Exception:
        return None

GH_TOKEN = get_secret('GH_TOKEN')
HF_TOKEN = get_secret('HF_TOKEN')

print('GH_TOKEN:', 'set' if GH_TOKEN else 'MISSING - the clone below will fail')
print('HF_TOKEN:', 'set' if HF_TOKEN else 'not set (unauthenticated, rate-limited)')

if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN

# Put the dataset cache in /tmp, NOT /kaggle/working. The Sangraha shards run to
# ~30 GB, which would exhaust the 20 GB persistent quota that has to hold the
# corpus and checkpoints. /tmp is wiped between sessions, so a resumed run
# re-downloads - that is the right trade: bandwidth is replaceable, quota is not.
os.environ['HF_HOME'] = '/tmp/hf_cache'
print('HF cache : /tmp/hf_cache  (not persisted - keeps the 20 GB quota for outputs)')


In [ ]:
REPO_URL = 'https://github.com/CL3-410/individual-project-Karan-thayat.git'
BRANCH   = 'phase-1'

if (REPO / '.git').exists():
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=False)
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=False)
    subprocess.run(['git', '-C', str(REPO), 'pull', 'origin', BRANCH], check=False)
    print('updated existing clone')
else:
    # The token is injected only at call time and scrubbed from any output, so it
    # never appears in the notebook or its saved results.
    auth_url = REPO_URL.replace('https://', f'https://{GH_TOKEN}@') if GH_TOKEN else REPO_URL
    result = subprocess.run(['git', 'clone', '--branch', BRANCH, auth_url, str(REPO)],
                            capture_output=True, text=True)
    text = result.stdout + result.stderr
    if GH_TOKEN:
        text = text.replace(GH_TOKEN, '***')
    print(text.strip() or 'cloned')

ok = (REPO / 'lma' / 'data' / 'sources.py').exists()
print('code present:', ok)
assert ok, 'Clone failed. Check GH_TOKEN has the classic `repo` scope.'


## 4 · Dependencies

Phase 1 needs no PyTorch. `sentencepiece` trains the tokenizers from scratch and
`datasets` is only a download client — neither is a pretrained model.


In [ ]:
%pip install -q sentencepiece datasets pyyaml beautifulsoup4 lxml regex 2>&1 | tail -2

import sentencepiece, datasets
print('sentencepiece', sentencepiece.__version__)
print('datasets     ', datasets.__version__)


## 5 · Scale the source manifests up

Two mechanical edits:

1. Widen the Sangraha shard pattern from one shard to all 77 — this is the change
   that takes the corpus from ~3.5M to hundreds of millions of tokens.
2. Drop the per-source document caps.

The annotated original manifest is preserved as `*.sampled.yaml`, because rewriting
YAML discards comments and those comments document why each source is configured
the way it is.


In [ ]:
import yaml

def scale_up(manifest_path, edits):
    """Widen shard globs and remove document caps for a full-scale run."""
    manifest_path = pathlib.Path(manifest_path)
    backup = manifest_path.with_suffix('.sampled.yaml')
    if not backup.exists():
        backup.write_text(manifest_path.read_text())

    cfg = yaml.safe_load(manifest_path.read_text())
    for src in cfg['sources']:
        if src['source_id'] not in edits:
            continue
        src.pop('limit', None)
        glob = edits[src['source_id']]
        if glob:
            src['data_files'] = glob
    manifest_path.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False))
    return cfg

cfg = scale_up(REPO / 'bengali/configs/sources.yaml', {
    'sangraha_bn':  'verified/ben/*.parquet',
    'wikipedia_bn': None,
})
for s in cfg['sources']:
    if s.get('enabled', True):
        print(f"  {s['source_id']:32s} {s.get('data_files') or s.get('config') or s.get('path','')}")


## 6 · Run the Bengali ingest

**The long cell.** Roughly 6–10 hours, which fits inside Kaggle's 12-hour session.

### If the session ends before it finishes

Re-run **this same cell** in a new session. `--resume` reconciles the output files
against the checkpoint, restores the deduplication index, and skips what each
source already yielded. `/kaggle/working` persists between sessions of the same
notebook, so the checkpoint is still there.

A resumed run has been verified to produce a byte-identical corpus to an
uninterrupted one, including after `kill -9`.

### Keep the tab open

An interactive Kaggle session stops when you close the browser. For an unattended
run use **Save Version → Save & Run All (Commit)**, which executes the whole
notebook in the background and keeps the outputs.


In [ ]:
cmd = [
    sys.executable, str(REPO / 'scripts/build_corpus.py'),
    '--config',  str(REPO / 'bengali/configs/pipeline.yaml'),
    '--sources', str(REPO / 'bengali/configs/sources.yaml'),
    '--profile', 'full',
    '--target-tokens', '500000000',
    '--checkpoint-dir', str(WORK / 'checkpoints_bn'),
    '--checkpoint-every', '25000',
    '--resume',          # safe on a first run: no checkpoint means start fresh
    '--report-every', '20000',
]
print(' '.join(cmd), '\n')

# Stream output live rather than buffering it to the end, so progress is visible.
proc = subprocess.Popen(cmd, cwd=str(REPO), stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
print('\nexit code:', proc.wait())


### Progress check

Run in a separate cell while the ingest is going, or in a new session to see how
far the previous one got.


In [ ]:
import json

state_file = WORK / 'checkpoints_bn' / 'bn_ingest_state.json'
if state_file.exists():
    st = json.load(open(state_file))
    budget = 500_000_000 * 4.0   # characters, at the assumed 4 chars/token
    print(f"segment     : {st['segments']}")
    print(f"docs seen   : {st['seen']:,}")
    print(f"docs kept   : {st['kept']:,}")
    print(f"train chars : {st['train_chars']:,}  ({100 * st['train_chars'] / budget:.1f}% of budget)")
    print(f"elapsed     : {st['elapsed_seconds'] / 3600:.1f} h")
else:
    print('No checkpoint - either the run has not saved one yet, or it finished and cleared it.')


## 7 · Nepali

Same code and same 500M target, but Nepali public text is roughly two orders of
magnitude scarcer, so this exhausts its sources and stops short. That is the
expected result; the assignment permits a documented shortfall and
`tokenize_corpus.py` reports the exact gap. Expect 1–2 hours.


In [ ]:
cfg = scale_up(REPO / 'nepali/configs/sources.yaml', {
    'sangraha_ne':  'verified/nep/*.parquet',
    'wikipedia_ne': None,
})

cmd = [
    sys.executable, str(REPO / 'scripts/build_corpus.py'),
    '--config',  str(REPO / 'nepali/configs/pipeline.yaml'),
    '--sources', str(REPO / 'nepali/configs/sources.yaml'),
    '--profile', 'full',
    '--target-tokens', '500000000',
    '--checkpoint-dir', str(WORK / 'checkpoints_ne'),
    '--checkpoint-every', '25000',
    '--resume',
]
proc = subprocess.Popen(cmd, cwd=str(REPO), stdout=subprocess.PIPE,
                        stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
print('\nexit code:', proc.wait())


## 8 · Tokenizers, token counts, verification, report


In [ ]:
def run(script, *extra):
    """Run a pipeline script in the repo, streaming its output."""
    cmd = [sys.executable, str(REPO / 'scripts' / script), *extra]
    proc = subprocess.Popen(cmd, cwd=str(REPO), stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    return proc.wait()

for name in ['bengali', 'nepali']:
    cfg_path = f'{name}/configs/pipeline.yaml'
    print(f'\n=== {name}: tokenizer sweep + training ===')
    run('train_tokenizer.py', '--config', cfg_path, '--profile', 'full',
        '--sweep', '--use-sweep-best')
    print(f'\n=== {name}: exact token count ===')
    run('tokenize_corpus.py', '--config', cfg_path)


In [ ]:
print('=== component self-tests ===')
run('test_pipeline.py')

print('\n=== independence validation ===')
rc = run('validate_independence.py')
print('exit code:', rc, '(0 = all checks passed)')

print('\n=== report ===')
run('make_report.py')

print('\n=== Phase 1 checklist ===')
run('check_phase1.py')


## 9 · Check the manual-collection share

⚠️ **This is the requirement most likely to fail at full scale.**

The bounded local run cleared 20% only because downloaded sources were capped.
Removing those caps multiplies the downloaded side roughly 100× while scraped
volume stays fixed, so the manual share collapses.

Three ways to close the gap:

- Raise `max_pages` on the Wikisource scrape sources (they cache, so re-runs are incremental)
- Enable the `manual_scrape_custom_*` templates with sites you are entitled to crawl
- OCR scanned books into `<lang>/data/raw/pdf/` and run `scripts/run_ocr.py`

Or keep a cap on the downloaded sources: a smaller corpus that satisfies the
requirement beats a larger one that does not.


In [ ]:
for name in ['bengali', 'nepali']:
    stats = REPO / name / 'reports/corpus_stats.json'
    if not stats.exists():
        continue
    d = json.load(open(stats))
    share = d['collection_split']['manual_word_fraction']
    verdict = 'OK' if share >= 0.20 else 'BELOW REQUIREMENT'
    print(f"{name:9s} manual {100 * share:5.1f}%  [{verdict}]")
    for src, words in sorted(d['by_source']['words'].items(), key=lambda kv: -kv[1]):
        print(f"    {src:32s} {words:>14,} words")


## 10 · Where the output lives, and how to get it out

Kaggle has no Drive mount. Persistence works differently, and the difference
matters for a multi-session run:

| Location | Survives the session? | Size | Purpose |
|---|---|---|---|
| `/kaggle/working/` | **Yes**, for *this notebook* | 20 GB | corpora, checkpoints, artifacts |
| `/tmp/` | No | — | dataset cache (deliberately disposable) |
| `/kaggle/input/` | Read-only | — | datasets you attach |

`/kaggle/working` is what makes `--resume` work across sessions: re-open **this
same notebook** and the checkpoint is still there. It is not a general filesystem
though — another notebook cannot read it.

### Getting files off Kaggle

**Small artifacts** (tokenizers, statistics, report — a few MB): the cell below
collects them, and **Save Version → Save & Run All (Commit)** makes them
downloadable from the notebook's Output tab. You can also push them straight to
GitHub with the last cell.

**Corpora** (several GB): too large for git. Two options —

- **Kaggle Dataset** — *Output tab → New Dataset*. Stays on Kaggle, gets a URL you
  can share, and can be attached to future notebooks as `/kaggle/input/...`.
- **Download, then upload to Drive** — more manual, but the assignment asks for
  Drive links in the README, so this is the path that satisfies the submission
  requirement directly.


In [ ]:
import shutil

OUT = WORK / 'phase1_artifacts'
OUT.mkdir(exist_ok=True)

# Small, graded artifacts - these belong in git.
for name in ['bengali', 'nepali']:
    for sub in ['tokenizer', 'reports', 'configs']:
        src = REPO / name / sub
        if src.exists():
            shutil.copytree(src, OUT / name / sub, dirs_exist_ok=True)
if (REPO / 'report').exists():
    shutil.copytree(REPO / 'report', OUT / 'report', dirs_exist_ok=True)

total_kb = 0
for path in sorted(OUT.rglob('*')):
    if path.is_file():
        kb = path.stat().st_size / 1024
        total_kb += kb
        print(f'{kb:9.1f} KB  {path.relative_to(OUT)}')
print(f'{total_kb / 1024:9.1f} MB  TOTAL (small enough for git)')

# Large corpora - copy into /kaggle/working so they survive the session and can be
# published as a Dataset or downloaded.
print('\nCorpora:')
CORPORA = WORK / 'corpora'
CORPORA.mkdir(exist_ok=True)
for name, lang in [('bengali', 'bn'), ('nepali', 'ne')]:
    src_dir = REPO / name / 'data/processed'
    if not src_dir.exists():
        continue
    for f in sorted(src_dir.glob('*.jsonl')):
        dest = CORPORA / f.name
        if not dest.exists() or dest.stat().st_size != f.stat().st_size:
            shutil.copy2(f, dest)
        print(f'  {dest.stat().st_size / 1e9:6.2f} GB  {dest}')

usage = shutil.disk_usage(WORK)
print(f'\n/kaggle/working: {usage.free / 1e9:.1f} GB free of {usage.total / 1e9:.1f} GB')
if usage.free < 2e9:
    print('  WARNING: low space. Delete the checkpoints once the run has completed.')

print('\nNext: Save Version -> Save & Run All (Commit) to persist these outputs.')
print('Then publish /kaggle/working/corpora as a Dataset, or download it for Drive.')


### Committing the corpora back to git

Only the small artifacts belong in git — tokenizers, statistics, the report. The
corpora go to Drive or a Kaggle Dataset, with the links in your README, as the
assignment requires.

Set your git identity to your own name and email before committing.


In [ ]:
# Optional: commit the small artifacts back to the phase-1 branch.
COMMIT = False   # set True when you want to push

if COMMIT and GH_TOKEN:
    subprocess.run(['git', '-C', str(REPO), 'config', 'user.name', 'Karan Thayat'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'config', 'user.email', 'karanthayat.30@gmail.com'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'add',
                    'bengali/reports', 'nepali/reports',
                    'bengali/tokenizer', 'nepali/tokenizer', 'report'], check=False)
    subprocess.run(['git', '-C', str(REPO), 'commit', '-m',
                    'Phase 1: full-scale corpus statistics and tokenizers'], check=False)
    push_url = REPO_URL.replace('https://', f'https://{GH_TOKEN}@')
    r = subprocess.run(['git', '-C', str(REPO), 'push', push_url, BRANCH],
                       capture_output=True, text=True)
    print((r.stdout + r.stderr).replace(GH_TOKEN, '***').strip())
else:
    print('COMMIT is False - nothing pushed.')
